In [63]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [65]:
df = pd.read_csv('06-trees/car_fuel_efficiency_2026.csv')
df.fillna(0, inplace=True)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [79]:
#Splits
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [80]:
#DictVectorizer
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=True)
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)


**Question 1**

Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
- Train a model with max_depth=1.

Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.
- 'vehicle_weight'
- 'model_year'
- 'origin'
- 'fuel_type'


In [81]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))


|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



**Question 2**

Train a random forest regressor with these parameters:

- n_estimators=10
- random_state=1
- n_jobs=-1 (optional - to make training faster)

What's the RMSE of this model on the validation data?

- 0.1837
- 1.837
- 18.37
- 183.7


In [84]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_pred, y_val)

1.8332197358745623

**Question 3**

Now let's experiment with the n_estimators parameter

- Try these values of this parameter: [10, 50, 100, 150].
- Set random_state to 1.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

- 10
- 50
- 100
- 150


In [87]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
    n_estimators=n,
    random_state=1,
    n_jobs=1
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_pred, y_val)
    print(f'For n_estimators={n}, rmse={rmse:.3f}')

For n_estimators=10, rmse=1.833
For n_estimators=50, rmse=1.774
For n_estimators=100, rmse=1.770
For n_estimators=150, rmse=1.769


**Question 4**

Let's select the best max_depth:

- Try different values of max_depth: [10, 15, 20, 25]
- For each of these values,
    - try different values of n_estimators from [10, 50, 100, 150]
    - calculate the mean RMSE
- Fix the random seed: random_state=1

What's the best max_depth, using the mean RMSE?

- 10
- 15
- 20
- 25


In [89]:
for d in [10, 15, 20, 25]:
    rmse = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
        n_estimators=n,
        max_depth=d,
        random_state=1,
        n_jobs=-1
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse.append(root_mean_squared_error(y_pred, y_val))
    print(f'For max_depth={d}, mean_rmse={np.mean(rmse):.3f}')

For max_depth=10, mean_rmse=1.754
For max_depth=15, mean_rmse=1.784
For max_depth=20, mean_rmse=1.787
For max_depth=25, mean_rmse=1.786


**Question 5**

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

- Train the model with these parameters:
    - n_estimators=10,
    - max_depth=20,
    - random_state=1,
    - n_jobs=-1 (optional)
- Get the feature importance information from this model

What's the most important feature (among these 4)?

- vehicle_weight
- horsepower
- acceleration
- engine_displacement


In [ ]:
dt = RandomForestRegressor(
    n_estimators = 10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
dt.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()

importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': dt.feature_importances_
}).sort_values(by='Importance', ascending=False)
importances
# vehicle_weight

,Feature,Importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998
14,origin=USA,0.042152
2,drivetrain=Front-wheel drive,0.020062
11,num_doors,0.019401
5,fuel_type=Diesel,0.016571


**Question 6**

Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:


In [99]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, 
                    label=y_train,
                    feature_names=features)

dval = xgb.DMatrix(X_val,
                   label=y_val,
                   feature_names=features)

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

evals_result = {}
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(xgb_params, dtrain, 
                  evals=watchlist,
                  num_boost_round=100,
                  evals_result=evals_result)
max(evals_result['val']['rmse'])

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512


[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	val-rmse:1.74388
[25]	train-rmse:1.35837	val-rmse:1.74359
[26]	train-rmse:1.35382	val-rmse:1.74430
[27]	train-rmse:1.34456	val-rmse:1.74461
[28]	train-rmse:1.34092	val-rmse:1.74330
[29]	train-rmse:1.33937	val-rmse:1.74470
[30]	train-rmse:1.33243	val-rmse:1.74548
[31]	train-rmse:1.32487	val-rmse:1.74536
[32]	train-rmse:1.31700	val-rmse:1.74756
[33]	train-rmse:1.30197	val-rmse:1.74850
[34]	train-rmse:1.29442	val-rmse:1.75124
[35]	train-rmse:1.29290	val-rmse:1.75198
[36]	train-rmse:1.28352	val-rmse:1.75352
[37]	train-rmse:1.27463	val-rmse:1.75648
[38]	train-rmse:1.26628	val-rmse:1.75766
[39]	train-rmse:1.25721	val-rmse:1.76267
[40]	train-rmse:

2.467818060057268

In [100]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, 
                    label=y_train,
                    feature_names=features)

dval = xgb.DMatrix(X_val,
                   label=y_val,
                   feature_names=features)

xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

evals_result = {}
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(xgb_params, dtrain, 
                  evals=watchlist,
                  num_boost_round=100,
                  evals_result=evals_result)
max(evals_result['val']['rmse'])

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va

2.7863543710404772